# Clinical Screening Robot — Qwen3-0.6B QLoRA Notebook



This Colab-ready notebook fine-tunes `Qwen/Qwen3-0.6B` with QLoRA via Unsloth on two symptom-to-disease datasets, evaluates the fine-tuned adapter against the base model, and exports a GGUF artifact for local inference.



> **Important:** This notebook is intended for research and screening assistance only. It preserves the training-time disclaimer verbatim in every target response and does **not** replace professional medical diagnosis.



## What this notebook does



- Installs and configures the Colab environment for Unsloth + Qwen3 fine-tuning.

- Downloads and standardizes two Kaggle symptom datasets.

- Builds a mixed thinking / non-thinking chat-format training set.

- Fine-tunes `unsloth/Qwen3-0.6B-unsloth-bnb-4bit` with LoRA.

- Reports Accuracy, Macro F1, Cohen's Kappa, confusion-matrix views, and BERTScore.

- Saves plots, metrics JSON, the LoRA adapter, and GGUF exports to Google Drive.


## Setup



This section installs pinned dependencies, checks GPU availability, sets seeds, and mounts Google Drive so artifacts survive Colab resets.


In [1]:
%%capture

import subprocess

import sys



def pip_install(*packages):

    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])



pip_install("--upgrade", "--force-reinstall", "--no-cache-dir", "unsloth", "unsloth_zoo")

pip_install(

    "--no-cache-dir",

    "bitsandbytes",

    "trl",

    "peft",

    "transformers>=4.51",

    "accelerate",

    "datasets",

    "kagglehub",

    "evaluate",

    "bert_score",

    "scikit-learn",

    "matplotlib",

    "seaborn",

    "pandas",

    "sentencepiece",

    "protobuf",

)


In [2]:
import gc

import glob

import json

import os

import random

import re

import shutil

import sys

import warnings

from pathlib import Path



import evaluate

import kagglehub

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

import seaborn as sns

import torch

from datasets import Dataset

from IPython.display import Markdown, display

from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, f1_score

from sklearn.model_selection import train_test_split

from transformers import set_seed

from trl import SFTConfig, SFTTrainer

from unsloth import FastLanguageModel

from unsloth.chat_templates import train_on_responses_only



warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")



SEED = 3407

MAX_SEQ_LENGTH = 2048

TRAIN_MAX_SEQ_LENGTH = 1024

MODEL_NAME = "unsloth/Qwen3-0.6B-unsloth-bnb-4bit"

DISCLAIMER_TEMPLATE = (

    "Based on the reported symptoms, the clinical indication points to: {disease}.\n"

    "Disclaimer: This is an AI auxiliary tool designed for healthcare professionals. "

    "It is not 100% precise and does not replace a professional medical diagnosis."

)

SYSTEM_PROMPT = (

    "You are a clinical AI assistant that supports healthcare professionals with preliminary "

    "symptom-based screening. Always answer with the standardized template exactly as trained. "

    "Do not claim certainty, do not prescribe treatment, and preserve the disclaimer."

)

NOTEBOOK_NAME = "clinical_screening_robot.ipynb"



random.seed(SEED)

np.random.seed(SEED)

set_seed(SEED)

if torch.cuda.is_available():

    torch.manual_seed(SEED)

    torch.cuda.manual_seed_all(SEED)



IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:

    from google.colab import drive

    drive.mount("/content/drive")

    DRIVE_OUT = "/content/drive/MyDrive/screening_robot"

else:

    DRIVE_OUT = os.path.abspath("artifacts/screening_robot")



PLOTS_OUT = os.path.join(DRIVE_OUT, "plots")

TMP_OUT = os.path.join(DRIVE_OUT, "tmp")

os.makedirs(DRIVE_OUT, exist_ok=True)

os.makedirs(PLOTS_OUT, exist_ok=True)

os.makedirs(TMP_OUT, exist_ok=True)



if torch.cuda.is_available():

    free_vram, total_vram = torch.cuda.mem_get_info()

    props = torch.cuda.get_device_properties(0)

    print(f"GPU: {torch.cuda.get_device_name(0)}")

    print(f"Compute capability: {props.major}.{props.minor}")

    print(f"Free VRAM: {free_vram / 2**30:.2f} GB")

    print(f"Total VRAM: {total_vram / 2**30:.2f} GB")

else:

    print("⚠️ No CUDA GPU detected. This notebook targets Google Colab with a T4 GPU.")



print(f"Artifacts directory: {DRIVE_OUT}")


AttributeError: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'

## Data



This section downloads both Kaggle datasets, standardizes them into a common instruction/input/output schema, caps Dataset 1 at 50 rows per disease, performs stratified train/test splits, and prepares Qwen3-style chat messages with a deterministic 75% thinking / 25% non-thinking mix.


In [ ]:
DATASET_1_HANDLE = "dhivyeshrk/diseases-and-symptoms-dataset"

DATASET_2_HANDLE = "niyarrbarman/symptom2disease"

DATASET_1_FILE = "Final_Augmented_dataset_Diseases_and_Symptoms.csv"

DATASET_2_FILE = "Symptom2Disease.csv"



def find_csv(root_dir, preferred_name=None):

    candidates = sorted(glob.glob(os.path.join(root_dir, "**", "*.csv"), recursive=True))

    if preferred_name:

        for path in candidates:

            if os.path.basename(path) == preferred_name:

                return path

    if not candidates:

        raise FileNotFoundError(f"No CSV files found under {root_dir}")

    return candidates[0]



try:

    ds1_dir = kagglehub.dataset_download(DATASET_1_HANDLE)

    ds2_dir = kagglehub.dataset_download(DATASET_2_HANDLE)

except Exception as exc:

    raise RuntimeError(

        "Kaggle download failed. If Colab prompts for authentication, upload or configure your Kaggle credentials and rerun the cell."

    ) from exc



ds1_csv = find_csv(ds1_dir, DATASET_1_FILE)

ds2_csv = find_csv(ds2_dir, DATASET_2_FILE)



print("Dataset 1:", ds1_csv)

print("Dataset 2:", ds2_csv)


In [ ]:
def clean_free_text(text):

    text = str(text).strip()

    text = re.sub(r"\s+", " ", text)

    return text



def normalize_label(text):

    text = clean_free_text(text).lower().replace("_", " ")

    text = re.sub(r"\s+", " ", text)

    return text



def normalize_symptom_name(text):

    text = str(text).strip().lower().replace("_", " ")

    text = re.sub(r"\s+", " ", text)

    return text



def is_active_symptom(value):

    if pd.isna(value):

        return False

    if isinstance(value, str):

        return value.strip().lower() in {"1", "true", "yes", "y", "present"}

    try:

        return float(value) > 0

    except Exception:

        return bool(value)



def pick_column(columns, preferred, fallback_idx=None):

    lowered = {col.lower(): col for col in columns}

    for option in preferred:

        if option.lower() in lowered:

            return lowered[option.lower()]

    if fallback_idx is not None:

        return columns[fallback_idx]

    raise KeyError(f"None of {preferred} found in columns: {list(columns)}")



def drop_rare_labels(df, label_col="output", min_count=2):

    counts = df[label_col].value_counts()

    keep = counts[counts >= min_count].index

    dropped = int((~df[label_col].isin(keep)).sum())

    filtered = df[df[label_col].isin(keep)].copy().reset_index(drop=True)

    return filtered, dropped



def cap_per_label(df, label_col="output", cap=50, seed=SEED):

    parts = []

    for label, group in df.groupby(label_col, sort=True):

        take = min(len(group), cap)

        parts.append(group.sample(n=take, random_state=seed))

    return pd.concat(parts, ignore_index=True)



def build_ds1(csv_path):

    raw = pd.read_csv(csv_path)

    disease_col = pick_column(list(raw.columns), ["diseases", "disease", "prognosis"], fallback_idx=0)

    symptom_cols = [col for col in raw.columns if col != disease_col]



    records = []

    for _, row in raw.iterrows():

        disease = clean_free_text(row[disease_col])

        symptoms = [normalize_symptom_name(col) for col in symptom_cols if is_active_symptom(row[col])]

        symptoms = [sym for sym in symptoms if sym]

        if not disease or not symptoms:

            continue

        records.append(

            {

                "instruction": "Given the symptoms, identify the disease.",

                "input": ", ".join(symptoms),

                "output": disease,

                "source": "diseases_and_symptoms",

            }

        )



    df = pd.DataFrame(records)

    df["normalized_output"] = df["output"].map(normalize_label)

    df, dropped = drop_rare_labels(df, label_col="normalized_output", min_count=2)

    if dropped:

        print(f"Dataset 1: dropped {dropped} rows belonging to labels with <2 examples before stratification.")

    df = cap_per_label(df, label_col="normalized_output", cap=50, seed=SEED)

    return df.reset_index(drop=True)



def build_ds2(csv_path):

    raw = pd.read_csv(csv_path)

    raw = raw.loc[:, ~raw.columns.str.contains(r"^Unnamed", case=False, regex=True)]

    text_col = pick_column(list(raw.columns), ["text", "symptoms", "sentence", "description"], fallback_idx=0)

    label_col = pick_column(list(raw.columns), ["label", "disease", "prognosis"], fallback_idx=1)



    df = pd.DataFrame(

        {

            "instruction": "Given the symptoms reported, identify the disease.",

            "input": raw[text_col].map(clean_free_text),

            "output": raw[label_col].map(clean_free_text),

            "source": "symptom2disease",

        }

    )

    df = df[(df["input"] != "") & (df["output"] != "")].copy().reset_index(drop=True)

    df["normalized_output"] = df["output"].map(normalize_label)

    df, dropped = drop_rare_labels(df, label_col="normalized_output", min_count=2)

    if dropped:

        print(f"Dataset 2: dropped {dropped} rows belonging to labels with <2 examples before stratification.")

    return df.reset_index(drop=True)



def stratified_split(df, test_size=0.1, seed=SEED):

    train_df, test_df = train_test_split(

        df,

        test_size=test_size,

        random_state=seed,

        stratify=df["normalized_output"],

    )

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)



def format_assistant(disease):

    return DISCLAIMER_TEMPLATE.format(disease=clean_free_text(disease))



def build_chat_messages(df, seed=SEED):

    rng = np.random.default_rng(seed)

    think_flags = rng.random(len(df)) < 0.75

    rows = []

    for think_flag, row in zip(think_flags, df.itertuples(index=False)):

        mode = "think" if think_flag else "no_think"

        user_content = (

            f"{row.instruction}\n\n"

            f"Reported symptoms:\n{row.input}\n\n"

            f"Respond using the standardized template only. {'/think' if think_flag else '/no_think'}"

        )

        assistant_content = format_assistant(row.output)

        if think_flag:

            assistant_content = (

                f"<think>\nThe reported symptoms most align with {clean_free_text(row.output)}.\n</think>\n\n"

                + assistant_content

            )

        messages = [

            {"role": "system", "content": SYSTEM_PROMPT},

            {"role": "user", "content": user_content},

            {"role": "assistant", "content": assistant_content},

        ]

        item = row._asdict()

        item["thinking_mode"] = mode

        item["messages"] = messages

        rows.append(item)

    return pd.DataFrame(rows)



ds1 = build_ds1(ds1_csv)

ds2 = build_ds2(ds2_csv)



train_ds1, test_ds1 = stratified_split(ds1)

train_ds2, test_ds2 = stratified_split(ds2)



train_ds1 = build_chat_messages(train_ds1, seed=SEED)

test_ds1 = build_chat_messages(test_ds1, seed=SEED + 1)

train_ds2 = build_chat_messages(train_ds2, seed=SEED + 2)

test_ds2 = build_chat_messages(test_ds2, seed=SEED + 3)



train_combined = pd.concat([train_ds1, train_ds2], ignore_index=True)

test_splits = {

    "diseases_and_symptoms": test_ds1.reset_index(drop=True),

    "symptom2disease": test_ds2.reset_index(drop=True),

}

test_combined = pd.concat(list(test_splits.values()), ignore_index=True)



assert len(train_combined) > 10000, f"Expected > 10000 training rows, found {len(train_combined)}"

assert not train_combined[["instruction", "input", "output", "source"]].isnull().any().any(), "Null values detected in training columns."

assert train_combined["input"].str.len().gt(0).mean() >= 0.90, "Less than 90% of inputs are non-empty."



summary_df = pd.DataFrame(

    [

        {"split": "train_ds1", "rows": len(train_ds1), "labels": train_ds1["normalized_output"].nunique()},

        {"split": "test_ds1", "rows": len(test_ds1), "labels": test_ds1["normalized_output"].nunique()},

        {"split": "train_ds2", "rows": len(train_ds2), "labels": train_ds2["normalized_output"].nunique()},

        {"split": "test_ds2", "rows": len(test_ds2), "labels": test_ds2["normalized_output"].nunique()},

        {"split": "train_combined", "rows": len(train_combined), "labels": train_combined["normalized_output"].nunique()},

        {"split": "test_combined", "rows": len(test_combined), "labels": test_combined["normalized_output"].nunique()},

    ]

)



display(summary_df)

display(train_combined.sample(3, random_state=SEED)[["source", "instruction", "input", "output", "thinking_mode"]])


## Training



This section loads the pre-quantized Qwen3 0.6B base model, applies LoRA adapters, materializes the chat-formatted `text` column via the tokenizer chat template, masks loss to assistant responses only, and fine-tunes on the combined dataset.


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(

    model_name=MODEL_NAME,

    max_seq_length=MAX_SEQ_LENGTH,

    dtype=None,

    load_in_4bit=True,

)



def render_chat_text(messages):

    return tokenizer.apply_chat_template(messages, tokenize=False)



for frame in [train_combined, test_combined, *test_splits.values()]:

    frame["text"] = frame["messages"].apply(render_chat_text)



sample_texts = train_combined["text"].sample(3, random_state=SEED).tolist()

for sample_text in sample_texts:

    assert "Based on the reported symptoms, the clinical indication points to:" in sample_text

    assert "<|im_start|>assistant" in sample_text



display(Markdown("### Spot-check: rendered chat examples"))

for sample_text in sample_texts:

    print(sample_text[:1000])

    print("-" * 100)



model = FastLanguageModel.get_peft_model(

    model,

    r=16,

    lora_alpha=32,

    lora_dropout=0.0,

    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],

    use_gradient_checkpointing="unsloth",

    random_state=SEED,

)



train_dataset = Dataset.from_pandas(train_combined[["text"]], preserve_index=False)

print(f"Training rows: {len(train_dataset):,}")

print(train_combined["thinking_mode"].value_counts(normalize=True).rename("ratio"))


In [ ]:
FastLanguageModel.for_training(model)



use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

max_steps = 400 if len(train_dataset) > 20000 else -1

INTERMEDIATE_LORA_DIR = os.path.join(DRIVE_OUT, "qwen3-clinical-lora")



training_args = SFTConfig(

    output_dir=os.path.join(TMP_OUT, "outputs"),

    per_device_train_batch_size=2,

    gradient_accumulation_steps=4,

    warmup_steps=20,

    num_train_epochs=1,

    max_steps=max_steps,

    learning_rate=2e-4,

    fp16=not use_bf16,

    bf16=use_bf16,

    logging_steps=20,

    optim="adamw_8bit",

    weight_decay=0.01,

    lr_scheduler_type="linear",

    seed=SEED,

    report_to="none",

    max_seq_length=TRAIN_MAX_SEQ_LENGTH,

)



trainer = SFTTrainer(

    model=model,

    tokenizer=tokenizer,

    train_dataset=train_dataset,

    dataset_text_field="text",

    max_seq_length=TRAIN_MAX_SEQ_LENGTH,

    packing=False,

    args=training_args,

)



trainer = train_on_responses_only(

    trainer,

    instruction_part="<|im_start|>user\n",

    response_part="<|im_start|>assistant\n",

)



train_result = trainer.train()



loss_history = [entry["loss"] for entry in trainer.state.log_history if "loss" in entry]

assert len(loss_history) >= 2, "Not enough loss points collected to evaluate training trend."

assert loss_history[-1] < loss_history[0], "Training loss did not end below the starting loss."



loss_df = pd.DataFrame({"step": range(1, len(loss_history) + 1), "loss": loss_history})

loss_plot_path = os.path.join(DRIVE_OUT, "training_loss.png")



fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(loss_df["step"], loss_df["loss"], marker="o", linewidth=2)

ax.set_title("Training loss")

ax.set_xlabel("Logged step")

ax.set_ylabel("Loss")

fig.tight_layout()

fig.savefig(loss_plot_path, dpi=200, bbox_inches="tight")

plt.show()



model.save_pretrained(INTERMEDIATE_LORA_DIR)

tokenizer.save_pretrained(INTERMEDIATE_LORA_DIR)

print(f"Saved intermediate LoRA adapter to: {INTERMEDIATE_LORA_DIR}")

print(f"Saved training-loss plot to: {loss_plot_path}")


## Evaluation



This section compares the fine-tuned adapter against the base model using qualitative examples, classification metrics, row-normalized confusion matrices, and BERTScore. To keep Colab runtimes sane, evaluation is sampled deterministically per split.


In [ ]:
def cleanup_memory():

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()



def extract_disease_label(text):

    match = re.search(r"clinical indication points to:\s*([^\.\n]+)\.", text, flags=re.IGNORECASE)

    if match:

        return clean_free_text(match.group(1))

    return clean_free_text(text)



def maybe_stratified_sample(df, target_n, label_col="normalized_output", seed=SEED):

    if len(df) <= target_n:

        return df.copy().reset_index(drop=True)

    target_n = max(target_n, df[label_col].nunique())

    try:

        sample_df, _ = train_test_split(

            df,

            train_size=target_n,

            random_state=seed,

            stratify=df[label_col],

        )

        return sample_df.reset_index(drop=True)

    except ValueError:

        return df.sample(n=target_n, random_state=seed).reset_index(drop=True)



def build_eval_messages(instruction, symptoms, thinking=False):

    suffix = "/think" if thinking else "/no_think"

    return [

        {"role": "system", "content": SYSTEM_PROMPT},

        {

            "role": "user",

            "content": (

                f"{instruction}\n\n"

                f"Reported symptoms:\n{symptoms}\n\n"

                f"Respond using the standardized template only. {suffix}"

            ),

        },

    ]



def generate_prediction(model_obj, tokenizer_obj, instruction, symptoms, thinking=False):

    messages = build_eval_messages(instruction, symptoms, thinking=thinking)

    prompt_text = tokenizer_obj.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True,

        enable_thinking=thinking,

    )

    model_inputs = tokenizer_obj([prompt_text], return_tensors="pt").to(model_obj.device)

    generation_kwargs = {

        "max_new_tokens": 160,

        "temperature": 0.6 if thinking else 0.7,

        "top_p": 0.95 if thinking else 0.8,

        "top_k": 20,

        "use_cache": True,

    }

    with torch.inference_mode():

        output_ids = model_obj.generate(**model_inputs, **generation_kwargs)

    new_tokens = output_ids[0][model_inputs["input_ids"].shape[1]:]

    decoded = tokenizer_obj.decode(new_tokens, skip_special_tokens=True).strip()

    return {

        "raw_output": decoded,

        "predicted_label": extract_disease_label(decoded),

    }



def run_predictions(model_obj, tokenizer_obj, df, thinking=False, tag="model"):

    rows = []

    for idx, row in enumerate(df.itertuples(index=False), start=1):

        result = generate_prediction(model_obj, tokenizer_obj, row.instruction, row.input, thinking=thinking)

        rows.append(

            {

                "source": row.source,

                "instruction": row.instruction,

                "input": row.input,

                "reference_raw": row.output,

                "reference_norm": row.normalized_output,

                "prediction_raw": result["raw_output"],

                "prediction_label": result["predicted_label"],

                "prediction_norm": normalize_label(result["predicted_label"]),

                "model": tag,

            }

        )

        if idx % 25 == 0:

            print(f"{tag}: generated {idx}/{len(df)} examples")

    return pd.DataFrame(rows)



def classification_metrics(pred_df):

    return {

        "accuracy": accuracy_score(pred_df["reference_norm"], pred_df["prediction_norm"]),

        "macro_f1": f1_score(pred_df["reference_norm"], pred_df["prediction_norm"], average="macro", zero_division=0),

        "cohen_kappa": cohen_kappa_score(pred_df["reference_norm"], pred_df["prediction_norm"]),

        "parse_rate": float((pred_df["prediction_label"].astype(str).str.len() > 0).mean()),

        "n": int(len(pred_df)),

    }



evaluation_splits = {

    "diseases_and_symptoms": maybe_stratified_sample(test_splits["diseases_and_symptoms"], 250, seed=SEED),

    "symptom2disease": maybe_stratified_sample(test_splits["symptom2disease"], 250, seed=SEED + 1),

    "combined": maybe_stratified_sample(test_combined, 300, seed=SEED + 2),

}



qualitative_samples = pd.concat(

    [

        evaluation_splits["diseases_and_symptoms"].sample(n=min(4, len(evaluation_splits["diseases_and_symptoms"])), random_state=SEED),

        evaluation_splits["symptom2disease"].sample(n=min(4, len(evaluation_splits["symptom2disease"])), random_state=SEED),

    ],

    ignore_index=True,

)

display(pd.DataFrame({k: [len(v)] for k, v in evaluation_splits.items()}))


In [ ]:
FastLanguageModel.for_inference(model)



fine_tuned_predictions = {}

for split_name, split_df in evaluation_splits.items():

    print(f"Running fine-tuned predictions for {split_name}...")

    fine_tuned_predictions[split_name] = run_predictions(model, tokenizer, split_df, thinking=False, tag="fine_tuned")



print("Running fine-tuned qualitative examples...")

qualitative_ft = run_predictions(model, tokenizer, qualitative_samples, thinking=False, tag="fine_tuned")



cleanup_memory()

base_model, base_tokenizer = FastLanguageModel.from_pretrained(

    model_name=MODEL_NAME,

    max_seq_length=MAX_SEQ_LENGTH,

    dtype=None,

    load_in_4bit=True,

)

FastLanguageModel.for_inference(base_model)



base_predictions = {}

for split_name, split_df in evaluation_splits.items():

    print(f"Running base-model predictions for {split_name}...")

    base_predictions[split_name] = run_predictions(base_model, base_tokenizer, split_df, thinking=False, tag="base")



print("Running base-model qualitative examples...")

qualitative_base = run_predictions(base_model, base_tokenizer, qualitative_samples, thinking=False, tag="base")



del base_model, base_tokenizer

cleanup_memory()



qualitative_table = qualitative_samples[["source", "input", "output"]].copy().rename(columns={"output": "reference"})

qualitative_table["fine_tuned_prediction"] = qualitative_ft["prediction_label"]

qualitative_table["base_prediction"] = qualitative_base["prediction_label"]

display(qualitative_table)



metric_rows = []

for split_name, pred_df in fine_tuned_predictions.items():

    scores = classification_metrics(pred_df)

    metric_rows.append({"split": split_name, "model": "fine_tuned", **scores})

for split_name, pred_df in base_predictions.items():

    scores = classification_metrics(pred_df)

    metric_rows.append({"split": split_name, "model": "base", **scores})



metrics_df = pd.DataFrame(metric_rows)

display(metrics_df.sort_values(["split", "model"]).reset_index(drop=True))



symptom2disease_ft = metrics_df.query("split == 'symptom2disease' and model == 'fine_tuned'").iloc[0]

symptom2disease_base = metrics_df.query("split == 'symptom2disease' and model == 'base'").iloc[0]

assert symptom2disease_ft["accuracy"] > symptom2disease_base["accuracy"], "Fine-tuned accuracy did not exceed base accuracy on Symptom2Disease."

assert symptom2disease_ft["cohen_kappa"] >= 0.6, "Fine-tuned Cohen's Kappa did not reach the ≥0.6 target on Symptom2Disease."


In [ ]:
SYSTEM_KEYWORDS = {

    "respiratory": ["asthma", "bronch", "pneum", "sinus", "tonsil", "cold", "flu", "influenza", "tb", "tuberculosis", "resp"],

    "cardiovascular": ["hypertension", "heart", "card", "stroke", "coronary", "angina"],

    "gastrointestinal": ["gastro", "stomach", "ulcer", "hepat", "liver", "jaundice", "typhoid", "diarr", "constipation"],

    "neurological": ["migraine", "paralysis", "epilep", "vertigo", "brain", "neuro"],

    "endocrine_metabolic": ["diabet", "thyroid", "obesity", "metabolic"],

    "dermatological": ["skin", "psoriasis", "eczema", "acne", "fungal", "allergy", "dermat"],

    "infectious_vector": ["malaria", "dengue", "chikungunya", "zika", "viral", "infection", "covid"],

    "musculoskeletal": ["arthritis", "joint", "bone", "muscle", "osteo"],

    "ophthalmic": ["eye", "conjunct", "vision", "ocular"],

    "urologic_reproductive": ["urinary", "kidney", "renal", "pregnan", "ovar", "prostate", "reproduct"],

}



def infer_system(label):

    label_norm = normalize_label(label)

    for system, keywords in SYSTEM_KEYWORDS.items():

        if any(keyword in label_norm for keyword in keywords):

            return system

    return "other"



def annotation_matrix(cm_norm, threshold=0.05):

    annot = np.empty(cm_norm.shape, dtype=object)

    for i in range(cm_norm.shape[0]):

        for j in range(cm_norm.shape[1]):

            annot[i, j] = f"{cm_norm[i, j]:.0%}" if cm_norm[i, j] >= threshold else ""

    return annot



def plot_confusion_heatmap(y_true, y_pred, labels, title, save_path):

    cm_norm = confusion_matrix(y_true, y_pred, labels=labels, normalize="true")

    valid_rows = cm_norm.sum(axis=1) > 0

    assert np.allclose(cm_norm[valid_rows].sum(axis=1), 1.0, atol=1e-6), f"Row-normalized confusion matrix failed for {title}"

    annot = annotation_matrix(cm_norm)

    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 0.45), max(6, len(labels) * 0.45)))

    sns.heatmap(

        cm_norm,

        cmap="mako",

        linewidths=0.25,

        linecolor="white",

        xticklabels=labels,

        yticklabels=labels,

        annot=annot,

        fmt="",

        cbar_kws={"label": "Row-normalized recall"},

        ax=ax,

    )

    ax.set_title(title)

    ax.set_xlabel("Predicted label")

    ax.set_ylabel("True label")

    plt.xticks(rotation=90)

    plt.yticks(rotation=0)

    fig.tight_layout()

    fig.savefig(save_path, dpi=220, bbox_inches="tight")

    plt.show()

    return cm_norm



def top_confusions(pred_df, top_k=10):

    labels = sorted(set(pred_df["reference_norm"]) | set(pred_df["prediction_norm"]))

    cm_norm = confusion_matrix(pred_df["reference_norm"], pred_df["prediction_norm"], labels=labels, normalize="true")

    rows = []

    for i, true_label in enumerate(labels):

        for j, pred_label in enumerate(labels):

            if i == j or cm_norm[i, j] <= 0:

                continue

            true_system = infer_system(true_label)

            pred_system = infer_system(pred_label)

            rows.append(

                {

                    "true_label": true_label,

                    "predicted_label": pred_label,

                    "rate": cm_norm[i, j],

                    "true_system": true_system,

                    "predicted_system": pred_system,

                    "confusion_type": "expected" if true_system == pred_system else "alarming",

                }

            )

    if not rows:

        return pd.DataFrame(columns=["true_label", "predicted_label", "rate", "true_system", "predicted_system", "confusion_type"])

    return pd.DataFrame(rows).sort_values("rate", ascending=False).head(top_k)



def plot_system_summary(pred_df, title, save_path):

    system_true = pred_df["reference_norm"].map(infer_system)

    system_pred = pred_df["prediction_norm"].map(infer_system)

    labels = sorted(set(system_true) | set(system_pred))

    cm_norm = confusion_matrix(system_true, system_pred, labels=labels, normalize="true")

    fig, ax = plt.subplots(figsize=(8, 6))

    sns.heatmap(cm_norm, annot=annotation_matrix(cm_norm, threshold=0.10), fmt="", cmap="crest", xticklabels=labels, yticklabels=labels, ax=ax)

    ax.set_title(title)

    ax.set_xlabel("Predicted system")

    ax.set_ylabel("True system")

    plt.xticks(rotation=45, ha="right")

    plt.yticks(rotation=0)

    fig.tight_layout()

    fig.savefig(save_path, dpi=220, bbox_inches="tight")

    plt.show()



def prepare_confusion_subset(pred_df, split_name):

    plot_df = pred_df.copy()

    if split_name == "symptom2disease":

        true_labels = sorted(plot_df["reference_norm"].value_counts().index.tolist())

    elif plot_df["reference_norm"].nunique() > 20:

        true_labels = plot_df["reference_norm"].value_counts().head(20).index.tolist()

        plot_df = plot_df[plot_df["reference_norm"].isin(true_labels)].copy()

    else:

        true_labels = sorted(plot_df["reference_norm"].value_counts().index.tolist())

    plot_df["prediction_for_plot"] = plot_df["prediction_norm"].where(plot_df["prediction_norm"].isin(true_labels), "<other>")

    plot_labels = true_labels + (["<other>"] if (plot_df["prediction_for_plot"] == "<other>").any() else [])

    return plot_df, plot_labels



def sanitize_for_json(value):

    if isinstance(value, dict):

        return {str(k): sanitize_for_json(v) for k, v in value.items()}

    if isinstance(value, list):

        return [sanitize_for_json(v) for v in value]

    if isinstance(value, tuple):

        return [sanitize_for_json(v) for v in value]

    if isinstance(value, np.generic):

        return value.item()

    return value



plot_metric_path = os.path.join(PLOTS_OUT, "metric_summary.png")

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)

for ax, metric_name, title in zip(

    axes,

    ["accuracy", "macro_f1", "cohen_kappa"],

    ["Accuracy", "Macro F1", "Cohen's Kappa"],

):

    sns.barplot(data=metrics_df, x="split", y=metric_name, hue="model", ax=ax)

    ax.set_title(title)

    ax.set_xlabel("")

    ax.tick_params(axis="x", rotation=20)

fig.tight_layout()

fig.savefig(plot_metric_path, dpi=220, bbox_inches="tight")

plt.show()



confusion_artifacts = {}

for split_name in ["diseases_and_symptoms", "symptom2disease", "combined"]:

    pred_df = fine_tuned_predictions[split_name]

    plot_df, plot_labels = prepare_confusion_subset(pred_df, split_name)

    heatmap_path = os.path.join(PLOTS_OUT, f"confusion_{split_name}.png")

    confusion_artifacts[split_name] = {

        "labels": plot_labels,

        "heatmap_path": heatmap_path,

        "top_confusions": top_confusions(plot_df, top_k=10),

    }

    plot_confusion_heatmap(

        plot_df["reference_norm"],

        plot_df["prediction_for_plot"],

        labels=plot_labels,

        title=f"Fine-tuned confusion matrix — {split_name}",

        save_path=heatmap_path,

    )

    display(confusion_artifacts[split_name]["top_confusions"])

    if split_name != "symptom2disease":

        system_path = os.path.join(PLOTS_OUT, f"confusion_system_{split_name}.png")

        plot_system_summary(plot_df, f"System-level confusion summary — {split_name}", system_path)

        confusion_artifacts[split_name]["system_heatmap_path"] = system_path



cleanup_memory()

bertscore_metric = evaluate.load("bertscore")



def compute_bertscore(pred_df, model_name):

    try:

        raw = bertscore_metric.compute(

            predictions=pred_df["prediction_label"].fillna("").tolist(),

            references=pred_df["reference_raw"].tolist(),

            lang="en",

        )

        scorer = "roberta-large"

    except RuntimeError as exc:

        if "out of memory" not in str(exc).lower():

            raise

        cleanup_memory()

        raw = bertscore_metric.compute(

            predictions=pred_df["prediction_label"].fillna("").tolist(),

            references=pred_df["reference_raw"].tolist(),

            lang="en",

            model_type="distilbert-base-uncased",

        )

        scorer = "distilbert-base-uncased"

    result = pd.DataFrame({

        "precision": raw["precision"],

        "recall": raw["recall"],

        "f1": raw["f1"],

        "model": model_name,

        "scorer": scorer,

    })

    return result



bertscore_ft = compute_bertscore(fine_tuned_predictions["combined"], "fine_tuned")

bertscore_base = compute_bertscore(base_predictions["combined"], "base")

bertscore_all = pd.concat([bertscore_ft, bertscore_base], ignore_index=True)



hist_path = os.path.join(PLOTS_OUT, "bertscore_f1_histogram.png")

fig, ax = plt.subplots(figsize=(10, 5))

sns.histplot(data=bertscore_all, x="f1", hue="model", bins=20, kde=True, stat="density", common_norm=False, ax=ax)

ax.set_title("BERTScore F1 distribution")

fig.tight_layout()

fig.savefig(hist_path, dpi=220, bbox_inches="tight")

plt.show()



bertscore_summary = bertscore_all.groupby("model")["f1"].agg(["mean", "std"]).reset_index()

display(bertscore_summary)

assert bertscore_summary.query("model == 'fine_tuned'")["mean"].iloc[0] > bertscore_summary.query("model == 'base'")["mean"].iloc[0], "Fine-tuned BERTScore F1 did not exceed base."



metrics_payload = {

    "config": {

        "seed": SEED,

        "model_name": MODEL_NAME,

        "max_seq_length": MAX_SEQ_LENGTH,

        "train_max_seq_length": TRAIN_MAX_SEQ_LENGTH,

    },

    "dataset_sizes": summary_df.to_dict(orient="records"),

    "classification_metrics": metrics_df.to_dict(orient="records"),

    "bertscore_summary": bertscore_summary.to_dict(orient="records"),

    "training_loss": loss_df.to_dict(orient="records"),

    "artifacts": {

        "training_loss_plot": loss_plot_path,

        "metric_summary_plot": plot_metric_path,

        "bertscore_histogram": hist_path,

        "confusion": {

            split_name: {

                key: value if not isinstance(value, pd.DataFrame) else value.to_dict(orient="records")

                for key, value in artifact.items()

            }

            for split_name, artifact in confusion_artifacts.items()

        },

    },

}



metrics_json_path = os.path.join(DRIVE_OUT, "metrics.json")

with open(metrics_json_path, "w", encoding="utf-8") as fp:

    json.dump(sanitize_for_json(metrics_payload), fp, indent=2)



print(f"Saved metrics JSON to: {metrics_json_path}")


## Export



This section persists the LoRA adapter, exports GGUF files for local inference, and attempts to copy the notebook itself into the Drive artifact directory.


In [ ]:
LORA_OUT = os.path.join(DRIVE_OUT, "qwen3-clinical-lora")

GGUF_OUT = os.path.join(DRIVE_OUT, "qwen3-clinical-gguf")

os.makedirs(GGUF_OUT, exist_ok=True)



model.save_pretrained(LORA_OUT)

tokenizer.save_pretrained(LORA_OUT)

print(f"LoRA adapter saved to: {LORA_OUT}")



model.save_pretrained_gguf(GGUF_OUT, tokenizer, quantization_method="q4_k_m")

print(f"Exported q4_k_m GGUF to: {GGUF_OUT}")



try:

    model.save_pretrained_gguf(GGUF_OUT, tokenizer, quantization_method="q8_0")

    print(f"Exported q8_0 GGUF to: {GGUF_OUT}")

except Exception as exc:

    print(f"Skipping q8_0 export due to: {exc}")



gguf_files = sorted(glob.glob(os.path.join(GGUF_OUT, "**", "*.gguf"), recursive=True))

assert gguf_files, "No GGUF files were produced."

for gguf_file in gguf_files:

    size_mb = os.path.getsize(gguf_file) / 2**20

    print(f"{gguf_file} -> {size_mb:.1f} MB")



copied = False

for candidate in [NOTEBOOK_NAME, os.path.join(os.getcwd(), NOTEBOOK_NAME)]:

    if os.path.exists(candidate):

        shutil.copy(candidate, os.path.join(DRIVE_OUT, NOTEBOOK_NAME))

        copied = True

        print(f"Copied notebook backup from {candidate} to {DRIVE_OUT}")

        break

if not copied:

    print("Notebook backup copy skipped automatically. Save a manual copy if the notebook file is not present in the runtime filesystem.")


### Local inference examples



Use the exported GGUF with `llama.cpp` or `llama-cpp-python`.



#### `llama-cli`



```bash

llama-cli \

  -m /path/to/qwen3-clinical-gguf/<your-file>.gguf \

  --ctx-size 4096 \

  --temp 0.7 \

  --top-p 0.8 \

  --top-k 20 \

  -p "<|im_start|>system\nYou are a clinical AI assistant that supports healthcare professionals with preliminary symptom-based screening. Always answer with the standardized template exactly as trained.\n<|im_end|>\n<|im_start|>user\nGiven the symptoms reported, identify the disease.\n\nReported symptoms:\nfever, headache, body pain, retro-orbital pain\n\nRespond using the standardized template only. /no_think\n<|im_end|>\n<|im_start|>assistant\n"

```



#### `llama-cpp-python`



```python

from llama_cpp import Llama



llm = Llama(

    model_path="/path/to/qwen3-clinical-gguf/<your-file>.gguf",

    n_ctx=4096,

    n_gpu_layers=-1,

    verbose=False,

)



prompt = """<|im_start|>system

You are a clinical AI assistant that supports healthcare professionals with preliminary symptom-based screening. Always answer with the standardized template exactly as trained.

<|im_end|>

<|im_start|>user

Given the symptoms reported, identify the disease.



Reported symptoms:

fever, headache, body pain, retro-orbital pain



Respond using the standardized template only. /no_think

<|im_end|>

<|im_start|>assistant

"""



result = llm(prompt, max_tokens=160, temperature=0.7, top_p=0.8, top_k=20, stop=["<|im_end|>"])

print(result["choices"][0]["text"])

```



Expected output should contain the disclaimer sentence:



`Disclaimer: This is an AI auxiliary tool designed for healthcare professionals. It is not 100% precise and does not replace a professional medical diagnosis.`
